# SENDAs Agreement 1 Update 2010-2022 (Prediction, step 2.3, internal validation)

<style type="text/css">
.showopt {
background-color: #004c93; color: #FFFFFF; width: 100px; height: 20px; text-align: center; vertical-align: middle !important; float: right; font-family: sans-serif; border-radius: 8px;
}
.showopt:hover {
background-color: #dfe4f2;
color: #004c93;
}
pre.plot {
background-color: white !important;
}
.tablelines table, .tablelines td, .tablelines th {
border: 1px solid black;
}
.centrado {
text-align: center;
}
.table.center {
margin-left:auto;
margin-right:auto;
}
/* https://vivekjaiskumar.medium.com/css-is-and-not-selector-17c942ec83f :is()*/
/* Applies to outputs that are not code other than R*/
pre {
overflow-x: auto !important;
}
pre code {
word-wrap: normal !important;
white-space: pre !important;
}
/*
pre:not(.sourceCode) {
white-space: nowrap !important;
}
*/
.sourceCode { /* Important gives precedence */
font-size: 10px !important;
line-height: 50% !important;
}
body{ /* Normal */
text-align: justify;
}
.superbigimage{
overflow-y:scroll;
height:350px;
white-space: nowrap;
overflow-x: auto;
width:100%;
}
.superbigimage img{
overflow-y: scroll;
overflow-x: hidden;
}
.message { color:#446C6E; font-family: monospace;font-size: 10px; line-height: 110%; font-weight: bold;}
div.blue { background-color:#e6f0ff; border-radius: 5px; padding: 5px; text-align: justify;}
div.red { background-color:#e6bab1; border-radius: 5px; padding: 5px; text-align: justify;}
.pandoc-table { /* Should add !important; but it seems no necessary */
margin-left:auto; /* To center */
margin-right:auto;
border-collapse: collapse;
table-layout: auto;
font-size: 11px;
overflow-y: auto;
max-height:450px !important;
white-space: nowrap;
overflow-x: auto;
width:450px;
}
.pandoc-table th {/* header */
text-align: center !important;
font-size: 10px;
padding: 0px;
}
.pandoc-table td {
text-align: left !important;
font-size: 9px;
padding: 0px;
}
.pandoc-table caption {
text-align: left !important;
font-size: 11px !important;
}
.center-table {
text-align: left !important;
font-size: 9px;
overflow-y:scroll;
height:450px;
overflow-x: scroll;
}
.controlly{
overflow-y:scroll;
height:350px;
overflow-x: scroll;
}
/*2025-03-07*/
.callout-warning,
.callout-error,
.callout-message {
  font-size: 0.7rem !important;  /* Adjust this value as needed */
}
.alert, .message {
    font-size: 0.7em !important;
}
.alert.alert-warning,
.alert.alert-danger,
.message {
    font-size: 0.7em !important;
}
</style>


<style>
#| label: css-format
h1 {
    color: var(--heading-color);
    font-size: 2rem;
    margin-bottom: 1vh;
}
p {
  font-size: 1.1rem;
  line-height: 1.6rem;
}
a {
  color: var(--primary-color);
  text-decoration: none;
  border-bottom: 3px solid transparent;
  font-weight: bold;
  &:hover, &:focus {
      border-bottom: 3px solid currentColor;
  }
}
section {
  margin: 0 auto;
}
.post-meta {
  font-size: 1rem;
  font-style: italic;
  display: block;
  margin-bottom: 4vh;
  color: var(--secondary-color);
}
nav {
  display: flex;
  justify-content: flex-end;
  padding: 20px 0;
}
/*slider switch css */
.theme-switch-wrapper {
  display: flex;
  align-items: center;
  
  em {
    margin-left: 10px;
    font-size: 1rem;
  }
}
.theme-switch {
  display: inline-block;
  height: 34px;
  position: relative;
  width: 60px;
}
.theme-switch input {
  display:none;
}
.slider {
  background-color: #ccc;
  bottom: 0;
  cursor: pointer;
  left: 0;
  position: absolute;
  right: 0;
  top: 0;
  transition: .4s;
}
.slider:before {
  background-color: #fff;
  bottom: 4px;
  content: "";
  height: 26px;
  left: 4px;
  position: absolute;
  transition: .4s;
  width: 26px;
}
input:checked + .slider {
  background-color: #66bb6a;
}
input:checked + .slider:before {
  transform: translateX(26px);
}
.slider.round {
  border-radius: 34px;
}
.slider.round:before {
  border-radius: 50%;
}
</style>


<style>
.scrollable-content {
  max-height: 350px;
  overflow-y: auto;
}
pre.scrollable-code {
  max-height: 350px;
  overflow-y: auto;
}
.superbigimage {
  overflow-x: scroll;
  white-space: nowrap;
}
.superbigimage img, 
.superbigimage svg {
  max-width: none;
  height: auto;
}
</style>
<br>

# Data Loading and Exploration

## Loading Packages and uniting databases

<div class="scrollable-content">


In [ ]:
#| label: setup
#| results: "hold"
#renv falls back to copying rather than symlinking, which is evidently very slow in this configuration.
renv::settings$use.cache(FALSE)
#only use explicit dependencies (in DESCRIPTION)
renv::settings$snapshot.type("implicit")
#check if rstools is installed
if(Sys.info()["sysname"]=="Windows"){
try(installr::install.Rtools(check_r_update=F))
}
check_quarto_version <- function(required = "1.7.29", comparator = c("ge","gt","le","lt","eq")) {
  comparator <- match.arg(comparator)
  current <- package_version(paste(unlist(quarto::quarto_version()), collapse = "."))
  req     <- package_version(required)
  ok <- switch(comparator,
               ge = current >= req,
               gt = current >  req,
               le = current <= req,
               lt = current <  req,
               eq = current == req)
  if (!ok) {
    stop(sprintf("Quarto version check failed: need %s %s (installed: %s).",
                 comparator, required, current), call. = FALSE)
  }
  invisible(TRUE)
}
check_quarto_version("1.7.29", "ge") 
#change repository to CL
local({
  r <- getOption("repos")
  r["CRAN"] <- "https://cran.dcc.uchile.cl/"
  options(repos=r)
})
if(!require(pacman)){install.packages("pacman");require(pacman)}
if(!require(pak)){install.packages("pak");require(pak)}
pacman::p_unlock(lib.loc = .libPaths()) #para no tener problemas reinstalando paquetes
if(Sys.info()["sysname"]=="Windows"){
if (getRversion() != "4.4.1") { stop("Requires R version 4.4.1; Actual: ", getRversion()) }
}
#check docker
check_docker_running <- function() {
  # Try running 'docker info' to check if Docker is running
  system("docker info", intern = TRUE, ignore.stderr = TRUE)
}
if(Sys.info()["sysname"]=="Windows"){
  install_docker <- function() {
    # Open the Docker Desktop download page in the browser for installation
    browseURL("https://www.docker.com/products/docker-desktop")
  }
  # Main logic
  if (inherits(try(check_docker_running(), silent = TRUE), "try-error")) {
    liftr::install_docker()
  } else {
    message("Docker is running.")
  }
}
#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_
#PACKAGES#######################################################################
#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_
unlink("*_cache", recursive=T)
pak::pak_sitrep()
# pak::sysreqs_check_installed(unique(unlist(paks)))
#pak::lockfile_create(unique(unlist(paks)),  "dependencies_duplicates24.lock", dependencies=T)
#pak::lockfile_install("dependencies_duplicates24.lock")
#https://rdrr.io/cran/pak/man/faq.html
#pak::cache_delete()
library(tidytable)
library(ggplot2)
library(readr)
library(tableone)
library(survivalmodels)
#renv::install("patchwork@1.2.0")
library(rms)
library(survidm)
library(caret)
library(survival)
library(riskRegression)
library(prodlim)
library(caret)
library(ggplot2)
library(dplyr)
library(survex)
library(pec)
library(future)
library(future.apply)
library(parallel)
library(mice)
library(riskRegression)
library(tidyr)
library(doFuture)
library(vcd)
library(survAUC)
library(knitr)
if (!requireNamespace("ipeval", quietly = TRUE)) {
  renv::install("ipeval")
}
#flexsurv
#if(!require(compareCstat)){install.pacakges("compareCstat")}
# library(shapr)#https://norskregnesentral.github.io/shapr/
# library(SemiMarkov)#https://www.degruyterbrill.com/document/doi/10.1515/ijb-2020-0083/html?lang=en
# #https://www.jclinepi.com/article/S0895-4356(24)00142-2/fulltext
#_#_#_#_#_#_#_#_#_#_#_#_#_-----------------------------
# 3. Activate polars code completion (safe to try even if it fails)
#_#_#_#_#_#_#_#_#_#_#_#_#_-----------------------------
#try(polars_code_completion_activate())
#_#_#_#_#_#_#_#_#_#_#_#_#_-----------------------------
# 4. BPMN from GitHub (not on CRAN, so install via devtools if missing)
#_#_#_#_#_#_#_#_#_#_#_#_#_-----------------------------
if (!requireNamespace("bpmn", quietly = TRUE)) {
  devtools::install_github("bergant/bpmn")
}
#_#_#_#_#_#_#_#_#_#_#_#_#_-----------------------------
# 5. PhantomJS Check (use webshot if PhantomJS is missing)
#_#_#_#_#_#_#_#_#_#_#_#_#_-----------------------------
# if (!webshot::is_phantomjs_installed()) {
#   webshot::install_phantomjs()
# }
#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_
#FUNCTIONS######################################################################
#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_
#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#
# NO MORE DEBUGS
options(error = NULL)        # si antes tenías options(error = recover) o browser)
options(browserNLdisabled = FALSE)
#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#
#NAs are replaced with "" in knitr kable
options(knitr.kable.NA = '')
pander::panderOptions('big.mark', ',')
pander::panderOptions('decimal.mark', '.')

#To produce line breaks in messages and warnings
knitr::knit_hooks$set(
   error = function(x, options) {
     paste('\n\n<div class="alert alert-danger" style="font-size: small !important;">',
           gsub('##', '\n', gsub('^##\ Error', '**Error**', x)),
           '</div>', sep = '\n')
   },
   warning = function(x, options) {
     paste('\n\n<div class="alert alert-warning" style="font-size: small !important;">',
           gsub('##', '\n', gsub('^##\ Warning:', '**Warning**', x)),
           '</div>', sep = '\n')
   },
   message = function(x, options) {
     paste('<div class="message" style="font-size: small !important;">',
           gsub('##', '\n', x),
           '</div>', sep = '\n')
   }
)

tnr<- "Times New Roman" 

options(scipen=2) #display numbers rather scientific number

# ── Helpers ────────────────────────────────────────────────────────────
mode_pick_int <- function(x){
  x <- x[!is.na(x)]
  if(length(x)==0) return(NA_integer_)
  tx <- sort(table(x), decreasing = TRUE)
  as.integer(names(tx)[1L])
}
subkey_to_label <- function(x){
  y <- gsub("_"," ", tolower(x))
  y <- gsub("amphetamine type stimulants","amphetamine-type stimulants", y)
  y <- gsub("tranquilizers hypnotics","tranquilizers/hypnotics", y)
  y
}

#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:
#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:
find_latest_file <- function(project_root, prefix) {
  # Get all files matching the pattern
  pattern <- paste0(prefix, "_\\d{8}_\\d{4}\\.csv$")
  files <- list.files(
    path = project_root,
    pattern = pattern,
    full.names = TRUE
  )
  if (length(files) == 0) {
    stop(paste("No files found matching pattern:", pattern))
  }
  # Extract timestamps from filenames (format: YYYYMMDD_HHMM)
  extract_timestamp <- function(filename) {
    matches <- regmatches(
      basename(filename),
      gregexpr("\\d{8}_\\d{4}", basename(filename))
    )
    if (length(matches[[1]]) == 0) {
      return(NA)
    }
    timestamp_str <- matches[[1]][length(matches[[1]])]
    return(timestamp_str)
  }
  timestamps <- sapply(files, extract_timestamp)
  # Remove any files where timestamp extraction failed
  valid_idx <- !is.na(timestamps)
  files <- files[valid_idx]
  timestamps <- timestamps[valid_idx]
  if (length(files) == 0) {
    stop("No valid timestamps found in filenames")
  }
  # Sort by timestamp (descending) and return the latest
  latest_idx <- order(timestamps, decreasing = TRUE)[1]
  return(files[latest_idx])
}

# Held-out (20%) internal validation

Final clean evaluation of the two primary models on the **retained 20% test split** (seed 2125). The 80% development set is used only to fit; the 20% test set is touched only for this out-of-sample evaluation. IBS, Uno's C-index, calibration and DCA use the
same estimators as `prediction225`; an `ipeval` bootstrap cross-check is added so discrimination/calibration can be checked for consistency across two independent methods.

Everything is driven by standalone scripts under `cons/_alt_scripts/` (see `VALIDATION_HOLDOUT_README.md`).
- **best_perf1** (SHAP primary): readmission `formula_shap_readmit_clean_updated` + mortality `formula_death_updated2`
- **best_perf2** (SHAP implemented): same readmission + mortality `formula_shap_death_rule2`, but as of 2026-06-13, with a second strata on physical diagnosis.

(The readmission model is identical in both, so readmission metrics are computed once.)

In [ ]:
#| label: define-project-root-and-paths
root <- here::here()
project_root <- gsub("/cons$", "", root)
data_out <- file.path(project_root, "data", "20241015_out")
figs_out <- file.path(root, "_figs")
out_dir  <- file.path(project_root, "cons", "_out")

In [ ]:
#| label: set-global-options
#| message: false

options(future.globals.maxSize = 10 * 1024^3)

In [ ]:
project_root <- gsub("/cons$", "", getwd())

In [ ]:
#| label: holdout-source-engines
#| message: false
# Reproducible data layer + all held-out engines. Sourcing run_validation_holdout.R
# pulls in build_holdout_datasets(), evaluate_dual_cox_holdout(), calibrate_*_holdout(),
# ipeval_holdout(), and the audited DCA engine (run_dca_full_summary / make_dca_panel_figure).
source(paste0(project_root,"/cons/_alt_scripts/val_holdout_01_rds_to_parquet.R"))   # corrected_datasets rds -> 5 parquet (idempotent)

In [ ]:
#| label: holdout-source-engines2
#| message: false

project_root <- gsub("/cons$", "", getwd())
source(paste0(project_root,"/","cons/_alt_scripts/run_validation_holdout.R"))

In [ ]:
#| label: holdout-build-sets
# Rebuild TRAIN (80%) / VALIDATION (20%) from the parquet layer + the seed-2125 split
# (cons/_out/comb_split_seed2125_test20_mar26.parquet). The rebuilt train is verified
# byte-identical to the authoritative py_corrected_datasets (last row of the table).
hd <- build_holdout_datasets(force = FALSE, verify = TRUE, verbose = TRUE)
train_list <- hd$train
val_list   <- hd$val
knitr::kable(hd$checks, "markdown",
  caption = "Held-out build consistency checks (train rebuilt from parquet + split)")


In [ ]:
#| label: holdout-define-models

formula_death_updated2 <- Surv(death_time_from_disch_m, death_event) ~ adm_age_rec3 + porc_pobr + 
    dit_m + national_foreign + ethnicity + dg_psiq_cie_10_instudy + 
    dg_psiq_cie_10_dg + dx_f3_mood + dx_f6_personality + dx_f_any_severe_mental + 
    any_phys_dx + polysubstance_strict + sex_rec_woman + cohabitation_family_of_origin + 
    cohabitation_with_couple_children + cohabitation_others + 
    sub_dep_icd10_status_drug_dependence + any_violence_1_domestic_violence_sex_abuse + 
    tr_outcome_referral + tr_outcome_dropout + tr_outcome_adm_discharge_adm_reasons + 
    adm_motive_sanitary_sector + adm_motive_another_sud_facility_fonodrogas_senda_previene + 
    adm_motive_justice_sector + adm_motive_other + first_sub_used_alcohol + 
    first_sub_used_cocaine_paste + first_sub_used_cocaine_powder + 
    first_sub_used_other + primary_sub_mod_cocaine_paste + primary_sub_mod_cocaine_powder + 
    primary_sub_mod_alcohol + primary_sub_mod_others + tipo_de_vivienda_rec2_other_unknown + 
    occupation_condition_corr24_unemployed + occupation_condition_corr24_inactive + 
    marital_status_rec_single + marital_status_rec_separated_divorced_annulled_widowed + 
    tenure_status_household_renting + tenure_status_household_others + 
    tenure_status_household_stays_temporarily_with_a_relative + 
    tenure_status_household_illegal_settlement + urbanicity_cat_2_mixed + 
    urbanicity_cat_1_rural + evaluacindelprocesoteraputico_logro_intermedio + 
    eva_consumo_logro_intermedio + eva_consumo_logro_minimo + 
    eva_fam_logro_intermedio + eva_fam_logro_minimo + eva_relinterp_logro_intermedio + 
    eva_relinterp_logro_minimo + eva_ocupacion_logro_intermedio + 
    eva_ocupacion_logro_minimo + eva_sm_logro_intermedio + eva_sm_logro_minimo + 
    eva_fisica_logro_intermedio + eva_fisica_logro_minimo + eva_transgnorma_logro_intermedio + 
    eva_transgnorma_logro_minimo + prim_sub_freq_rec_2_2_6_days_wk + 
    prim_sub_freq_rec_3_daily + ed_attainment_corr_2_completed_high_school_or_less + 
    ed_attainment_corr_3_completed_primary_school_or_less + strata(plan_type_strata) + 
    strata(tr_outcome_adm_discharge_rule_violation_undet)

formula_shap_death_rule2 <- Surv(death_time_from_disch_m, death_event) ~ adm_age_rec3 + primary_sub_mod_cocaine_paste + 
    primary_sub_mod_cocaine_powder + primary_sub_mod_alcohol + 
    primary_sub_mod_others + prim_sub_freq_rec_2_2_6_days_wk + 
    prim_sub_freq_rec_3_daily + occupation_condition_corr24_unemployed + 
    occupation_condition_corr24_inactive + eva_ocupacion_logro_intermedio + 
    eva_ocupacion_logro_minimo + cohabitation_family_of_origin + 
    cohabitation_with_couple_children + cohabitation_others + 
    strata(plan_type_strata) + strata(any_phys_dx)

f_readmit <- Surv(readmit_time_from_disch_m, readmit_event) ~ primary_sub_mod_cocaine_paste + 
    primary_sub_mod_cocaine_powder + primary_sub_mod_alcohol + 
    primary_sub_mod_others + adm_age_rec3 + porc_pobr + sex_rec_woman + 
    ethnicity + dit_m + eva_consumo_logro_intermedio + eva_consumo_logro_minimo + 
    ed_attainment_corr_2_completed_high_school_or_less + ed_attainment_corr_3_completed_primary_school_or_less + 
    occupation_condition_corr24_unemployed + occupation_condition_corr24_inactive + 
    dg_psiq_cie_10_dg + sub_dep_icd10_status_drug_dependence + 
    polysubstance_strict + eva_sm_logro_intermedio + eva_sm_logro_minimo + 
    evaluacindelprocesoteraputico_logro_intermedio + 
    tr_outcome_referral + tr_outcome_dropout + tr_outcome_adm_discharge_adm_reasons + 
    prim_sub_freq_rec_2_2_6_days_wk + prim_sub_freq_rec_3_daily + 
    strata(plan_type_strata) + strata(tr_outcome_adm_discharge_rule_violation_undet)


models <- list(
  best_perf1 = list(readmit = f_readmit, death = formula_death_updated2),  # SHAP readmit + Full PH death
  best_perf2 = list(readmit = f_readmit, death = formula_shap_death_rule2)        # SHAP readmit + SHAP death
)
EVAL_TIMES   <- c(3, 6, 12, 36, 60)  # C-index/IBS-over-time grid, 2026-03-18
DCA_HORIZONS <- c(6, 12, 36, 60)
CAL_TIMES    <- c(6, 12, 36, 60)


## C-index and IBS (held-out 20%)

Fit on TRAIN imputation *i*, predict on VALIDATION imputation *i*, pool across the 5 imputations. Reuses `.dual_cox_fit_one_risk` (Uno's C via `survival::concordance(timewt="n/G2", reverse=TRUE)`; IBS via IPCW) from `evaluate_dual_cox_python_style_boot.R`.

In [ ]:
models$best_perf1$readmit

In [ ]:
models$best_perf1$death

In [ ]:
models$best_perf2$readmit

In [ ]:
models$best_perf2$death

In [ ]:
#| label: holdout-cindex-ibs-run
results_boot_val_bp1 <- evaluate_dual_cox_holdout(
  models$best_perf1$readmit, models$best_perf1$death,
  train_list, val_list, eval_times = EVAL_TIMES)
results_boot_val_bp2 <- evaluate_dual_cox_holdout(
  models$best_perf2$readmit, models$best_perf2$death,
  train_list, val_list, eval_times = EVAL_TIMES)


In [ ]:
#| label: holdout-cindex-ibs-global
cindex_ibs_global <- dplyr::bind_rows(
  dplyr::mutate(dplyr::filter(results_boot_val_bp1$summary, Time == "Global"), model = "best_perf1"),
  dplyr::mutate(dplyr::filter(results_boot_val_bp2$summary, Time == "Global"), model = "best_perf2")
) |>
  dplyr::select(model, Risk, Metric, mean, sd, q025, q975)
knitr::kable(cindex_ibs_global, "markdown", digits = 4,
  caption = "Held-out 20%: global Uno's C-index and IBS (mean/sd across imputations)")


In [ ]:
#| label: holdout-cindex-ibs-plot
# NOTE: error bars are the across-imputation spread (tiny: outcomes are shared across
# imputations). Sampling uncertainty is given by the ipeval bootstrap CIs further below.
set1_colors <- rev(RColorBrewer::brewer.pal(3, "Set1")[1:2])
plot_metrics <- function(s, ttl) {
  dplyr::filter(s, Time != "Global") |>
    dplyr::mutate(Time_num = as.numeric(as.character(Time)),
                  Risk = factor(Risk, levels = c("Readmission", "Death")),
                  Metric = ifelse(Metric == "Uno's C-Index", "Discrimination (C-index)", "Prediction error (IBS)")) |>
    ggplot(aes(Time_num, mean, color = Risk, fill = Risk)) +
    geom_line(linewidth = 1) + geom_point(size = 1.5) +
    geom_errorbar(aes(ymin = q025, ymax = q975), width = 2, alpha = 0.6) +
    facet_wrap(~Metric, scales = "free_y") +
    scale_color_manual(values = set1_colors) + scale_fill_manual(values = set1_colors) +
    scale_x_continuous(breaks = seq(0, 108, 12)) +
    labs(x = "Months since discharge", y = "Value (95% CI)", color = "Outcome", fill = "Outcome",
         title = ttl) +
    theme_classic(base_size = 14) + theme(legend.position = "bottom")
}
plot_metrics(results_boot_val_bp1$summary, NULL)#"best_perf1 (SHAP readmit + Full PH death)")
plot_metrics(results_boot_val_bp2$summary, NULL)#"best_perf2 (SHAP readmit + SHAP death)")


## Calibration (held-out 20%)

Readmission: `riskRegression::CSC` → `predictRisk(cause=1)` → **Aalen-Johansen** CIF within deciles + loess ICI (span 0.75). Mortality: `coxph` → `predictRisk` → **Kaplan-Meier** within deciles + loess ICI. ECE is the n-weighted bin error; E:O is mean predicted over overall observed. Same math as `calibrate_cscox_aj_grouped_corr.R` / `calibrate_death_km_grouped.R`, applied to held-out predictions and pooled over imputations.

In [ ]:
#| label: holdout-calibration-run
#| message: false
cal_readmit   <- calibrate_readmit_holdout(f_readmit, train_list, val_list, times = CAL_TIMES)  # shared model
cal_death_bp1 <- calibrate_death_holdout(models$best_perf1$death, train_list, val_list, times = CAL_TIMES)
cal_death_bp2 <- calibrate_death_holdout(models$best_perf2$death, train_list, val_list, times = CAL_TIMES)


In [ ]:
#| label: holdout-calibration-tables
show_cal <- function(x, lab) dplyr::mutate(
  x$pooled_summary[, c("time_months", "ici_mean", "ici_p025", "ici_p975", "ece_mean", "eo_mean", "mean_pred", "observed")],
  model = lab)
dplyr::bind_rows(
  show_cal(cal_readmit,   "Readmission (shared)"),
  show_cal(cal_death_bp1, "Mortality best_perf1 (Full PH)"),
  show_cal(cal_death_bp2, "Mortality best_perf2 (SHAP)")
) |> knitr::kable("markdown", digits = 4,
     caption = "Held-out 20%: ICI / ECE / E:O by horizon (pooled over imputations)")


In [ ]:
#| label: holdout-calibration-plot
cal_curve_plot <- function(cal_obj, ttl, color) {
  d <- cal_obj$calibration_curves
  d$facet <- factor(paste0(d$time_months, " months"),
                    levels = paste0(sort(unique(d$time_months)), " months"))
  ggplot(d, aes(mean_predicted, observed)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", alpha = 0.6) +
    geom_ribbon(aes(ymin = observed_lower, ymax = observed_upper), alpha = 0.25, fill = color) +
    geom_line(color = color) + geom_point(color = color, size = 1.1) +
    facet_wrap(~facet, nrow = 1) +
    labs(x = "Predicted risk", y = "Observed (AJ / KM)", title = ttl) +
    theme_bw(base_size = 12)
}
cal_curve_plot(cal_readmit,  NULL, "#2166AC") #"Readmission calibration (Aalen-Johansen, held-out 20%)"
cal_curve_plot(cal_death_bp1, NULL,    "#B2182B") #"Mortality calibration best_perf1 (KM, held-out 20%)"
cal_curve_plot(cal_death_bp2, NULL,    "#B2182B") #"Mortality calibration best_perf2 (KM, held-out 20%)"


In [ ]:
#| label: holdout-cal-fig-functions
library(patchwork); library(scales)
tnr <- "Times New Roman"

# --- prediction225 "Epidemiology" calibration panel (curve + CI + density sidebars) ---
create_cal_published <- function(data, time_title, color = "#2166AC",
                                 x_lim = c(0, 0.4), x_by = 0.1, y_lim = x_lim, y_by = x_by,
                                 show_x_label = FALSE, show_y_label = FALSE) {
  fixed_margin <- ggplot2::margin(5, 2, 5, 5)
  .label_fmt <- function(lim) scales::number_format(accuracy =
    ifelse(diff(lim) <= 0.02, 0.001, ifelse(diff(lim) <= 0.2, 0.01, 0.1)))
  main <- ggplot2::ggplot(data, ggplot2::aes(mean_predicted, observed)) +
    ggplot2::geom_abline(intercept = 0, slope = 1, linetype = "dashed", linewidth = 0.4, alpha = 0.6) +
    ggplot2::geom_ribbon(ggplot2::aes(ymin = observed_lower, ymax = observed_upper), alpha = 0.3, fill = color) +
    ggplot2::geom_line(color = color, linewidth = 0.9) +
    ggplot2::geom_errorbar(ggplot2::aes(ymin = observed_lower, ymax = observed_upper),
                           width = 0.02 * diff(x_lim), color = color, alpha = 0.6, linewidth = 0.5) +
    ggplot2::geom_point(size = 1, color = color, fill = "white", shape = 21, stroke = 0.8) +
    ggplot2::scale_x_continuous(limits = x_lim, expand = c(0, 0),
                                breaks = seq(x_lim[1], x_lim[2], x_by), labels = .label_fmt(x_lim)) +
    ggplot2::scale_y_continuous(limits = y_lim, expand = c(0, 0),
                                breaks = seq(y_lim[1], y_lim[2], y_by), labels = .label_fmt(y_lim)) +
    ggplot2::labs(x = "Predicted probability", y = "Observed proportion", subtitle = time_title) +
    ggplot2::theme_bw(base_size = 15, base_family = tnr) +
    ggplot2::theme(
      plot.subtitle = ggplot2::element_text(size = 13, face = "bold", hjust = 0.5),
      panel.grid.major = ggplot2::element_line(color = "grey90", linewidth = 0.3),
      panel.grid.minor = ggplot2::element_blank(),
      axis.text = ggplot2::element_text(size = 13, color = "black"),
      axis.title = ggplot2::element_text(size = 13, face = "bold"),
      axis.title.x = ggplot2::element_text(color = if (show_x_label) "black" else NA, size = 13, face = "bold"),
      axis.title.y = ggplot2::element_text(color = if (show_y_label) "black" else NA, size = 13, face = "bold"),
      plot.margin = fixed_margin)
  right_dens <- ggplot2::ggplot(data, ggplot2::aes(observed)) +
    ggplot2::geom_density(fill = color, alpha = 0.3, color = NA) +
    ggplot2::scale_x_continuous(limits = y_lim, expand = c(0, 0)) +
    ggplot2::scale_y_continuous(expand = ggplot2::expansion(mult = c(0, 0.25))) +
    ggplot2::coord_flip() + ggplot2::theme_void() + ggplot2::theme(plot.margin = ggplot2::margin(5,5,5,0))
  bottom_dens <- ggplot2::ggplot(data, ggplot2::aes(mean_predicted)) +
    ggplot2::geom_density(fill = color, alpha = 0.3, color = NA) +
    ggplot2::scale_x_continuous(limits = x_lim, expand = c(0, 0)) +
    ggplot2::scale_y_continuous(expand = ggplot2::expansion(mult = c(0, 0.25))) +
    ggplot2::theme_void() + ggplot2::theme(plot.margin = ggplot2::margin(0,2,0,5))
  design <- c(patchwork::area(1, 1, 10, 10), patchwork::area(1, 11, 10, 11), patchwork::area(11, 1, 11, 10))
  main + right_dens + bottom_dens + patchwork::plot_layout(design = design)
}
.make_cal_panel <- function(cal_obj, color, x_lim, x_by, y_lim = x_lim, y_by = x_by) {
  curves <- dplyr::mutate(cal_obj$calibration_curves, time_label = paste0(time_months, " months"))
  tls <- paste0(sort(unique(curves$time_months)), " months")
  plots <- lapply(seq_along(tls), function(i)
    create_cal_published(dplyr::filter(curves, time_label == tls[i]), tls[i], color,
      x_lim, x_by, y_lim, y_by, show_x_label = (i == ceiling(length(tls)/2)), show_y_label = (i == 1)))
  patchwork::wrap_plots(plots, ncol = 4)
}

# --- IMPROVED: clean multi-horizon curve with the calibration indices annotated ---
# 2026-06-22
# Use the improved, parametrized calibration-curve function (shared .R) instead of the
# inline definition above; this also provides assemble_cal_curves() for the stacked figure.
source(file.path(if (exists("project_root")) project_root else getwd(),
                 "cons/_alt_scripts/cal_fig_improved.R"))
# para treams independientes y reproducibles por worker sin tener que clavar un número distinto a mano en cada uno

In [ ]:
#| label: holdout-cal-fig-improved
#| fig-width: 11
#| fig-height: 8
# Three calibration-curve rows (readmission, mortality Full PH, mortality SHAP) in ONE
# figure with shared x/y titles, A/B/C tags and the calibration indices annotated.
g_cal_curves <- assemble_cal_curves(
  rows = list(
    list(cal = cal_readmit,   color = "#2166AC", x_max = 0.40, ttl = "Readmission (Aalen-Johansen)"),
    list(cal = cal_death_bp1, color = "#B2182B", x_max = 0.15, ttl = "Mortality - all predictors (Full PH)"),
    list(cal = cal_death_bp2, color = "#B2182B", x_max = 0.15, ttl = "Mortality - SHAP-informed")),
  lang = "en", tnr = "Times New Roman",
  file = file.path(figs_out, "holdout_calibration_curves.png"),
  width = 28, height = 21)
print(g_cal_curves)

In [ ]:
#| label: holdout-cal-fig-epi
#| fig-width: 13
#| fig-height: 9
# Readmission (single row, shared model)
print(.make_cal_panel(cal_readmit, color = "#2166AC", x_lim = c(0, 0.4), x_by = 0.1))

# Mortality A = best_perf1 (Full PH), B = best_perf2 (SHAP) — Epidemiology A/B panel
death_A <- .make_cal_panel(cal_death_bp1, "#B2182B", x_lim = c(0, 0.15), x_by = 0.05)
death_B <- .make_cal_panel(cal_death_bp2, "#B2182B", x_lim = c(0, 0.15), x_by = 0.05)
final_death <- patchwork::wrap_plots(
  patchwork::wrap_elements(death_A & ggplot2::theme(axis.title.x = ggplot2::element_blank())),
  patchwork::wrap_elements(death_B & ggplot2::theme(axis.title.x = ggplot2::element_blank())),
  ncol = 1) +
  patchwork::plot_annotation(tag_levels = "A", caption = "Predicted probability",
    theme = ggplot2::theme(
      plot.caption = ggplot2::element_text(size = 13, face = "bold", hjust = 0.5, family = tnr),
      plot.tag = ggplot2::element_text(face = "bold", size = 18, family = tnr)))
print(final_death)
ggplot2::ggsave(file.path(figs_out, "holdout_mortality_calibration_AB.tiff"),
                final_death, width = 17.8 * 1.6, height = 12 * 1.6, units = "cm",
                dpi = 600, compression = "lzw")


## Decision Curve Analysis (held-out 20%)

Reuses the audited engine `run_adca_from_results_boot` via `run_dca_full_summary` on the held-out `$raw_predictions`: **Aalen-Johansen** observed risk for readmission (death competing), **1−KM** for mortality. Net-benefit math unchanged from the original.

In [ ]:
#| label: holdout-dca-run
#| message: false
dca_models_full <- run_dca_full_summary(
  list(best_perf1 = results_boot_val_bp1, best_perf2 = results_boot_val_bp2),
  horizons = DCA_HORIZONS)
dca_nb_bp1 <- summarize_dca_nb(dca_models_full$best_perf1$summary)
dca_nb_bp2 <- summarize_dca_nb(dca_models_full$best_perf2$summary)


In [ ]:
#| label: holdout-dca-table
knitr::kable(dca_nb_bp1$any_useful, "markdown", digits = 4,
  caption = "best_perf1 (held-out 20%): does the model beat treat-all & treat-none at any threshold?")


In [ ]:
#| label: holdout-dca-plot
# Row A = best_perf1 (Full PH death), Row B = best_perf2 (SHAP death)
make_dca_panel_figure(dca_models_full$best_perf1$summary,
                      dca_models_full$best_perf2$summary,
                      outcome = "readmission", horizons = c(6, 12, 36, 60))
make_dca_panel_figure(dca_models_full$best_perf1$summary,
                      dca_models_full$best_perf2$summary,
                      outcome = "death", horizons = c(6, 12, 36, 60))


In [ ]:
#| label: holdout-dca-panel-figures-std
#| message: false
# Standardized-net-benefit DCA panels (held-out 20%) via the shared .R helper.
# Standardized NB = net benefit / observed event rate at each horizon (1.0 = treat-all at
# threshold 0); x truncated per outcome from the focus windows.
source(file.path(if (exists("project_root")) project_root else getwd(),
                 "cons/_alt_scripts/make_dca_panel_figure_std.R"))
stopifnot(exists("dca_models_full"))
DCA_HZ_STD <- c(6, 12, 36, 60)

# Per-outcome two-row panels. Readmission rows A/B are identical (shared SHAP readmit);
# the real contrast is death: A = Full PH (best_perf1) vs B = SHAP (best_perf2).
p_readmit_panel_hold <- make_dca_panel_figure_std(
  summary_full = dca_models_full$best_perf1$summary,
  summary_ml   = dca_models_full$best_perf2$summary,
  outcome = "readmission", horizons = DCA_HZ_STD, lang = "en")
print(p_readmit_panel_hold)

p_death_panel_hold <- make_dca_panel_figure_std(
  summary_full = dca_models_full$best_perf1$summary,
  summary_ml   = dca_models_full$best_perf2$summary,
  outcome = "death", horizons = DCA_HZ_STD, lang = "en",
  row_labels = c(A = "All\npredictors", B = "SHAP-\ninformed"))
print(p_death_panel_hold)

# The three meaningful panels in ONE figure: A = readmission (shared), B = mortality all
# predictors, C = mortality SHAP-informed. A/B/C tags, shared y, x per row, one legend.
p_dca_abc <- make_dca_panel_figure_abc(
  dca_models_full, horizons = DCA_HZ_STD, lang = "en", tnr = "Times New Roman")
print(p_dca_abc)

fig_dir <- file.path(if (exists("project_root")) project_root else getwd(), "cons", "_figs")
if (!dir.exists(fig_dir)) dir.create(fig_dir, recursive = TRUE)
for (nm in c("p_dca_holdout_readmit_bp1_vs_bp2", "p_dca_holdout_death_bp1_vs_bp2", "p_dca_holdout_abc")) {
  obj <- switch(nm,
    p_dca_holdout_readmit_bp1_vs_bp2 = p_readmit_panel_hold,
    p_dca_holdout_death_bp1_vs_bp2   = p_death_panel_hold,
    p_dca_holdout_abc                = p_dca_abc)
  wd <- if (nm == "p_dca_holdout_abc") 26 else 17.8 * 1.2
  ht <- if (nm == "p_dca_holdout_abc") 18 else 12 * 1.2
  ggplot2::ggsave(file.path(fig_dir, paste0(nm, ".png")), obj, width = wd, height = ht, units = "cm", dpi = 600)
  ggplot2::ggsave(file.path(fig_dir, paste0(nm, ".pdf")), obj, width = wd, height = ht, units = "cm",
                  device = grDevices::cairo_pdf)
}
message(attr(p_dca_abc, "caption"))

In [ ]:
#| label: holdout-dca-per1000
#| message: false
# Absolute "per 1000" translation of the HELD-OUT DCA, with replicate CIs.
.per1000_holdout <- function(sm, rk, thr_show, horizons = c(12, 36, 60)) {
  d <- sm[sm$risk == rk & sm$strategy == "Model" & sm$horizon %in% horizons, , drop = FALSE]
  f1 <- function(v) formatC(round(v, 1), format = "f", digits = 1)
  ci <- function(m, lo, hi) sprintf("%s (%s, %s)", f1(m), f1(lo), f1(hi))
  out <- do.call(rbind, lapply(thr_show, function(t) {
    r <- d[abs(d$threshold - t) < 1e-9, , drop = FALSE]; if (!nrow(r)) return(NULL)
    data.frame(
      horizon = r$horizon, thr = paste0(t * 100, "%"),
      in_focus = ifelse(t >= r$focus_lower & t <= r$focus_upper, "*", ""),
      event_per1000 = f1(r$observed_event_risk_mean * 1000),
      captured_vs_none = ci(r$net_benefit_mean*1000, r$net_benefit_q025*1000, r$net_benefit_q975*1000),
      avoided_vs_all  = ci(r$interventions_avoided_mean*10, r$interventions_avoided_q025*10, r$interventions_avoided_q975*10),
      stringsAsFactors = FALSE)
  }))
  out[order(out$horizon), ]
}
thr_re <- c(0.05, 0.10, 0.15, 0.20, 0.30)
thr_de <- c(0.01, 0.02, 0.03, 0.05, 0.10)
cat("== READMISSION (best_perf1 == best_perf2; per 1000) ==\n")
print(.per1000_holdout(dca_models_full$best_perf1$summary, "readmission", thr_re), row.names = FALSE)
cat("\n== DEATH A = best_perf1 (Full PH) per 1000 ==\n")
print(.per1000_holdout(dca_models_full$best_perf1$summary, "death", thr_de), row.names = FALSE)
cat("\n== DEATH B = best_perf2 (SHAP) per 1000 ==\n")
print(.per1000_holdout(dca_models_full$best_perf2$summary, "death", thr_de), row.names = FALSE)
cat("\ncaptured_vs_none = net true cases caught /1000 (vs treat-none);",
    "avoided_vs_all = unnecessary interventions avoided /1000 (vs treat-all).\n")


### Bootstrap

stdca (KM/AJ risk in the positive subsample)

In [ ]:
#| label: holdout-dca-bootstrap-per1000
#| message: false
suppressPackageStartupMessages({ library(survival) })
stopifnot(exists(".extract_replicates"), exists(".compute_replicate_curves"),
          exists("results_boot_val_bp1"), exists("results_boot_val_bp2"))

# ---- config ----
B_BOOT      <- 1000L                       # smoke-test first with 50, then 1000-2000
HZ_BOOT     <- c(12, 36, 60)               # add 6 if it renders cleanly on held-out
THR_RE      <- sort(c(0.05, 0.10, 0.15, 0.20, 0.30))
THR_DE      <- sort(c(0.01, 0.02, 0.03, 0.05, 0.10))
BOOT_SEED   <- 2125L
USE_PARALLEL<- TRUE
ADCA_SOURCE <- file.path(if (exists("project_root")) project_root else getwd(),
                         "cons", "_alt_scripts", "adca_from_results_boot.R")

# ---- pool the 5 imputation blocks (avg predicted risk; outcomes shared row-wise) ----
.pool_blocks <- function(blocks) {
  nr <- nrow(blocks[[1]]$pred_risk)
  stopifnot(all(vapply(blocks, function(b) nrow(b$pred_risk) == nr, logical(1))))
  b0 <- blocks[[1]]
  b0$pred_risk <- Reduce(`+`, lapply(blocks, `[[`, "pred_risk")) / length(blocks)
  b0$replicate_id <- 1L
  b0
}

# ---- bootstrap one pooled block -> per-1000 CI table ----
.run_boot <- function(block, thresholds, horizons, method, B, seed) {
  thresholds <- sort(thresholds)
  n <- nrow(block$pred_risk)
  keep <- c("horizon","threshold","nb_model","nb_treat_all","interventions_avoided","observed_event_risk")
  pt <- .compute_replicate_curves(block, thresholds, horizons, readmit_method = method)[, keep]
  pk <- paste(pt$horizon, pt$threshold, sep = "_")

  set.seed(seed)
  idx <- lapply(seq_len(B), function(i) sample.int(n, n, replace = TRUE))

  one <- function(bi) {
    if (!exists(".compute_replicate_curves", mode = "function")) {
      suppressPackageStartupMessages(library(survival)); source(ADCA_SOURCE)
    }
    b <- block
    b$pred_risk <- b$pred_risk[bi, , drop = FALSE]; b$time <- b$time[bi]; b$event <- b$event[bi]
    if (!is.null(b$cr_ftime))   b$cr_ftime   <- b$cr_ftime[bi]
    if (!is.null(b$cr_fstatus)) b$cr_fstatus <- b$cr_fstatus[bi]
    d <- .compute_replicate_curves(b, thresholds, horizons, readmit_method = method)
    k <- paste(d$horizon, d$threshold, sep = "_")
    list(capt = setNames(d$nb_model * 1000, k),
         evit = setNames(d$interventions_avoided * 10, k))
  }
  reps <- if (USE_PARALLEL && requireNamespace("future.apply", quietly = TRUE)) {
    future.apply::future_lapply(idx, one, future.seed = TRUE)
  } else lapply(idx, one)

  cm <- sapply(reps, function(r) r$capt[pk]); em <- sapply(reps, function(r) r$evit[pk])
  if (is.null(dim(cm))) { cm <- matrix(cm, nrow = length(pk)); em <- matrix(em, nrow = length(pk)) }
  q <- function(M, p) apply(M, 1, quantile, probs = p, na.rm = TRUE, names = FALSE)
  prev <- pt$observed_event_risk
  data.frame(
    horizon = pt$horizon, threshold = pt$threshold,
    in_focus = ifelse(pt$threshold >= 0.5 * prev & pt$threshold <= 2 * prev, "*", ""),
    event_per1000 = round(prev * 1000, 1),
    capt_pt = pt$nb_model * 1000, capt_lo = q(cm, .025), capt_hi = q(cm, .975),
    evit_pt = pt$interventions_avoided * 10, evit_lo = q(em, .025), evit_hi = q(em, .975)
  )
}

# ---- optional parallel backend (defensive: half the cores, sequential startup) ----
if (USE_PARALLEL && requireNamespace("future", quietly = TRUE)) {
  # KEY FIX: start workers one-by-one (avoids the "X of Y workers failed to connect" timeout)
  options(parallelly.makeNodePSOCK.setup_strategy = "sequential")
  options(parallelly.makeNodePSOCK.connectTimeout = 600)  # seconds, generous for Drive paths

  N_WORKERS <- max(1L, parallelly::availableCores() %/% 2L)   # half the cores
  N_WORKERS <- min(N_WORKERS, 8L)                             # defensive cap; raise if stable

  ok <- tryCatch({ future::plan(future::multisession, workers = N_WORKERS); TRUE },
                 error = function(e) {
                   message("Parallel setup failed (", conditionMessage(e),
                           ") -> falling back to sequential."); FALSE })
  if (!ok) { USE_PARALLEL <- FALSE; future::plan(future::sequential) }
  message("DCA bootstrap workers: ", if (ok) N_WORKERS else 0L)
}

# ---- extract + pool once (readmit shared bp1==bp2; death from bp1 & bp2) ----
re_pool  <- .pool_blocks(.extract_replicates(results_boot_val_bp1, "readmission",
                                             attach_competing = TRUE, competing_risk = "death"))
de1_pool <- .pool_blocks(.extract_replicates(results_boot_val_bp1, "death"))
de2_pool <- .pool_blocks(.extract_replicates(results_boot_val_bp2, "death"))

boot_re  <- .run_boot(re_pool,  THR_RE, HZ_BOOT, "aalen-johansen", B_BOOT, BOOT_SEED)
boot_de1 <- .run_boot(de1_pool, THR_DE, HZ_BOOT, "km",             B_BOOT, BOOT_SEED)
boot_de2 <- .run_boot(de2_pool, THR_DE, HZ_BOOT, "km",             B_BOOT, BOOT_SEED)

# ---- print ----
.ci <- function(m, lo, hi) sprintf("%.1f (%.1f, %.1f)", m, lo, hi)
.show <- function(d, lab) {
  cat("\n== ", lab, " | per 1000 | bootstrap B=", B_BOOT, " (percentile 95% CI) ==\n", sep = "")
  o <- d[order(d$horizon, d$threshold), ]
  print(data.frame(horizon = o$horizon, thr = paste0(o$threshold * 100, "%"), foc = o$in_focus,
                   `event/1000` = o$event_per1000,
                   `captados_vs_none` = .ci(o$capt_pt, o$capt_lo, o$capt_hi),
                   `evitados_vs_all`  = .ci(o$evit_pt, o$evit_lo, o$evit_hi),
                   check.names = FALSE), row.names = FALSE)
}
.show(boot_re,  "READMISSION (best_perf1 == best_perf2)")
.show(boot_de1, "DEATH A = best_perf1 (Full PH)")
.show(boot_de2, "DEATH B = best_perf2 (SHAP)")
cat("\ncaught_vs_none = net true cases caught per 1,000 vs treat-none;",
    "avoided_vs_all = unnecessary interventions avoided per 1,000 vs treat-all.",
    "CI = nonparametric bootstrap of the held-out test patients (model frozen, no refit).\n")

~15 minutes

- Mortality, 36 months, 3% threshold: ~12 deaths detected per 1,000 patients (95% CI 9.8-14.4), plus ~442 unnecessary interventions avoided per 1,000 (390-493).

- Readmission, 36 months, 20% threshold: ~38 readmissions detected per 1,000 patients (32-44), plus ~152 unnecessary interventions avoided per 1,000 (130-174).

- Honest boundary: the readmission benefit over treat-all disappears below ~10-15% (the CI for avoided interventions crosses 0); cite only within the focal window (`foc = *`).

## Reclassification: NRI & IDI (held-out 20%)

Competing-risk-aware reclassification (readmission via Aalen-Johansen IPCW, mortality via standard IPCW), mirroring `prediction225`. `old` = SHAP implemented death (`best_perf2`), `new` = Full PH primary death (`best_perf1`): positive NRI/IDI = the Full PH primary reclassifies better than the parsimonious implementation. Readmission is shared by both models, so its NRI/IDI is 0 by construction — the **mortality** rows carry the signal. NRI/IDI is a secondary metric; its intervals are split-to-split percentiles, not CIs.

In [ ]:
#| label: holdout-nri-idi-run
#| message: false

source("cons/_alt_scripts/nri_idi_from_results_boot.R")          # AJ competing-aware
source("cons/_hist_scripts/make_nri_idi_epidemiology_table.R")

NRI_IDI_HORIZONS   <- c(6, 12, 36, 60)
NRI_IDI_CUT_POINTS <- c(0.05, 0.10, 0.20)

nri_idi_holdout <- run_nri_idi_from_results_boot(
  results_boot_old = results_boot_val_bp2,   # SHAP implemented death
  results_boot_new = results_boot_val_bp1,   # Full PH primary death
  horizons       = NRI_IDI_HORIZONS,
  cut_points     = NRI_IDI_CUT_POINTS,
  old_label      = "SHAP implemented (best_perf2)",
  new_label      = "Full PH primary (best_perf1)",
  readmit_method = "aalen-johansen",
  output_dir     = file.path(out_dir, "nri_idi_holdout"),
  prefix         = "nri_idi_holdout",
  save_raw       = FALSE
)
cat(sprintf("Readmission CR=%d | IPCW fallback=%d\n",
            nri_idi_holdout$config$n_readmit_competing_risk,
            nri_idi_holdout$config$n_readmit_ipcw_fallback))

nri_idi_tab <- make_nri_idi_epidemiology_table(
  reclass_object = nri_idi_holdout,
  output_dir = file.path(out_dir, "nri_idi_epi_holdout"),
  prefix     = "nri_idi_epi_holdout",
  old_label  = "SHAP implemented (best_perf2)",
  new_label  = "Full PH primary (best_perf1)"
)


source(file.path(gsub("/cons","",getwd()),
  "cons",
  "_alt_scripts",
  "relabel_nri_idi_table.R")
)
nri_idi_fixed <- relabel_nri_idi_table(
  nri_idi_tab$table,
  old_label = "SHAP implemented (best_perf2)",   # referencia
  new_label = "Full PH primary (best_perf1)"     # actualizado
)
nri_idi_model_table <- nri_idi_fixed$model_table

utils::write.csv(nri_idi_model_table,
                 file.path(gsub("/cons","",getwd()), "table_nri_holdout.csv"), row.names= FALSE)

In [ ]:
#| label: holdout-nri-idi-table
nri_idi_fixed$table |>
  knitr::kable("markdown", caption = nri_idi_fixed$caption)

## ipeval bootstrap consistency check (held-out 20%)

`ipeval::ip_score` with the dummy-treatment IPW trick (random 50/50 → weight 2). Strata-correct `predictRisk` vectors are fed directly (ipeval's internal `predict_cox` mishandles stratified Cox). Cross-check: ipeval `auc` ↔ Uno's C-index, `oeratio` ↔ E:O, `brier` ↔ point-horizon Brier. Requires `renv::install("ipeval")`.

In [ ]:
#| label: holdout-resource-engines
if (!exists("project_root")) project_root <- gsub("/cons$", "", here::here())
source(file.path(project_root, "cons/_alt_scripts/validate_holdout_metrics.R"))  # -> ici_bootstrap_holdout()
source(file.path(project_root, "cons/_alt_scripts/validate_holdout_ipeval.R"))   # -> ipeval_holdout(parallel=)
"parallel" %in% names(formals(ipeval_holdout))   # ipeval_holdout now takes parallel
exists("ici_bootstrap_holdout")                  # the ICI bootstrap is loaded


In [ ]:
#| label: holdout-ipeval-run
#| message: false
library(future); library(future.apply)
if (!inherits(future::plan(), "multisession")) future::plan(future::multisession, workers = 20)
options(future.globals.maxSize = 10 * 1024^3)

B_BOOT <- 1000L
ipeval_res <- if (requireNamespace("ipeval", quietly = TRUE)) {
  ipeval_holdout(models, train_list, val_list, horizons = DCA_HORIZONS,
                 bootstrap = B_BOOT, seed = 2125L, parallel = TRUE, verbose = TRUE)
} else { message("ipeval not installed; run renv::install('ipeval') to enable."); NULL }


~ 13 minutes

In [ ]:
#| label: holdout-ipeval-table
if (!is.null(ipeval_res)) {
  knitr::kable(ipeval_res$pooled, "markdown", digits = 3,
    caption = "ipeval (held-out 20%, dummy-treatment IPW): AUC / Brier / O:E with bootstrap 95% CI")
}

In [ ]:
#| label: holdout-ipeval-plot
#| fig-width: 11
#| fig-height: 4.5
tnr <- "Times New Roman"

plot_ipeval_epi <- function(ipeval_res, tnr = "Times New Roman") {
  p <- ipeval_res$pooled
  mk <- function(m, lab) data.frame(
    risk = p$risk, model = p$model, horizon = p$horizon, metric = lab,
    value = p[[paste0(m, "_mean")]],
    lower = p[[paste0(m, "_lower_mean")]], upper = p[[paste0(m, "_upper_mean")]],
    stringsAsFactors = FALSE)
  long <- rbind(mk("auc", "AUC(t)"), mk("brier", "Brier(t)"))
  lab_map <- c("readmit::best_perf1" = "Readmission (shared)",
               "death::best_perf1"   = "Mortality best_perf1 (Full PH)",
               "death::best_perf2"   = "Mortality best_perf2 (SHAP)")
  long$model_lab <- factor(unname(lab_map[long$model]), levels = unname(lab_map))
  long$metric    <- factor(long$metric, levels = c("AUC(t)", "Brier(t)"))
  long <- long[order(long$model_lab, long$horizon), ]   # ensure lines connect correctly
  pd <- position_dodge(width = 2.5)
  href <- data.frame(metric = factor("AUC(t)", levels = levels(long$metric)),
                     y = 0.5)                          # chance line
  cols <- c("Readmission (shared)"           = "#2166AC",
            "Mortality best_perf1 (Full PH)" = "#B2182B",
            "Mortality best_perf2 (SHAP)"    = "#E08214") #best_perf2
  ggplot2::ggplot(long, ggplot2::aes(horizon, value, color = model_lab, fill = model_lab, group = model_lab)) +
    ggplot2::geom_hline(data = href, ggplot2::aes(yintercept = y),
                        linetype = "dashed", color = "grey50", linewidth = 0.4) +
    ggplot2::geom_ribbon(ggplot2::aes(ymin = lower, ymax = upper), alpha = 0.15, color = NA, position = pd) +
    ggplot2::geom_line(linewidth = 0.8, position = pd) +
    ggplot2::geom_point(size = 1.8, position = pd) +
    ggplot2::geom_errorbar(ggplot2::aes(ymin = lower, ymax = upper),
                           width = 1.2, linewidth = 0.4, alpha = 0.7, position = pd) +
    ggplot2::facet_wrap(~metric, scales = "free_y", nrow = 1) +
    ggplot2::scale_color_manual(values = cols) + ggplot2::scale_fill_manual(values = cols) +
    ggplot2::scale_x_continuous(breaks = sort(unique(long$horizon))) +
    ggplot2::labs(x = "Months since discharge", y = "Value (bootstrap 95% CI)",
                  color = NULL, fill = NULL,
                  title = NULL)+#"ipeval (held-out 20%): discrimination (AUC) and accuracy (Brier)") +
    ggplot2::theme_bw(base_size = 13, base_family = tnr) +
    ggplot2::theme(legend.position = "bottom", panel.grid.minor = ggplot2::element_blank(),
                   strip.background = ggplot2::element_rect(fill = "grey95", color = "grey80"),
                   strip.text = ggplot2::element_text(face = "bold"))
}

if (!is.null(ipeval_res)) {
  g_ip <- plot_ipeval_epi(ipeval_res, tnr)
  print(g_ip)
  ggplot2::ggsave(file.path(figs_out, "holdout_ipeval_bootstrap.png"),
                  g_ip, width = 28, height = 11, units = "cm", dpi = 600)
}

In [ ]:
#| label: holdout-cindex-bootstrap
#| message: false
suppressPackageStartupMessages({ library(survival) })
stopifnot(exists(".dual_cox_safe_concordance"),   # sourced via the dual evaluator
          exists("results_boot_val_bp1"), exists("results_boot_val_bp2"))

B_CIDX            <- 1000L                 # smoke-test with 50 first
HZ_CIDX           <- c(6, 12, 36, 60)         # plus "Global"; add 6 if you want
CIDX_SEED         <- 2125L
USE_PARALLEL_CIDX <- TRUE
EVAL_SOURCE <- file.path(if (exists("project_root")) project_root else getwd(),
                         "cons", "_alt_scripts", "evaluate_dual_cox_python_style_boot.R")

# ---- pooled linear predictor + shared outcomes from raw_predictions ----
.extract_lp <- function(results_boot, risk) {
  lp <- list(); tt <- NULL; ev <- NULL
  for (item in results_boot$raw_predictions) {
    b <- item[[risk]]
    if (!is.list(b) || !is.null(b$error) || is.null(b$lp_val) || is.null(b$y_val)) next
    lp[[length(lp) + 1L]] <- as.numeric(b$lp_val)
    if (is.null(tt)) { tt <- as.numeric(b$y_val$time); ev <- as.integer(b$y_val$event) }
  }
  stopifnot(length(lp) >= 1L, all(lengths(lp) == length(tt)))
  list(lp = Reduce(`+`, lp) / length(lp), time = tt, event = ev, n_imp = length(lp))
}

# ---- bootstrap Uno's C (timewt = n/G2) for one frozen score ----
.run_cidx_boot <- function(dat, horizons, B, seed) {
  n <- length(dat$lp)
  uno <- function(bi) {
    t <- dat$time[bi]; e <- dat$event[bi]; s <- dat$lp[bi]
    c(Global = .dual_cox_safe_concordance(t, e, s),
      setNames(vapply(horizons, function(h) .dual_cox_safe_concordance(t, e, s, ymax = h),
                      numeric(1)), as.character(horizons)))
  }
  pt <- uno(seq_len(n))
  set.seed(seed)
  idx <- lapply(seq_len(B), function(i) sample.int(n, n, replace = TRUE))
  one <- function(bi) {
    if (!exists(".dual_cox_safe_concordance", mode = "function")) {
      suppressPackageStartupMessages(library(survival)); source(EVAL_SOURCE)
    }
    t <- dat$time[bi]; e <- dat$event[bi]; s <- dat$lp[bi]
    c(Global = .dual_cox_safe_concordance(t, e, s),
      setNames(vapply(horizons, function(h) .dual_cox_safe_concordance(t, e, s, ymax = h),
                      numeric(1)), as.character(horizons)))
  }
  reps <- if (USE_PARALLEL_CIDX && requireNamespace("future.apply", quietly = TRUE)) {
    future.apply::future_lapply(idx, one, future.seed = TRUE)
  } else lapply(idx, one)
  M <- do.call(cbind, reps)
  data.frame(metric = names(pt),
             Uno_C = sprintf("%.4f (%.4f, %.4f)", pt,
                             apply(M, 1, quantile, .025, na.rm = TRUE, names = FALSE),
                             apply(M, 1, quantile, .975, na.rm = TRUE, names = FALSE)),
             stringsAsFactors = FALSE)
}

# ---- defensive parallel backend (same fix as the DCA bootstrap) ----
if (USE_PARALLEL_CIDX && requireNamespace("future", quietly = TRUE) &&
    !inherits(future::plan(), "multisession")) {
  options(parallelly.makeNodePSOCK.setup_strategy = "sequential")
  options(parallelly.makeNodePSOCK.connectTimeout = 600)
  nw <- min(8L, max(1L, parallelly::availableCores() %/% 2L))
  ok <- tryCatch({ future::plan(future::multisession, workers = nw); TRUE },
                 error = function(e) { message("seq fallback: ", conditionMessage(e)); FALSE })
  if (!ok) { USE_PARALLEL_CIDX <- FALSE; future::plan(future::sequential) }
  message("Uno's C bootstrap workers: ", if (ok) nw else 0L)
}

re_dat  <- .extract_lp(results_boot_val_bp1, "readmission")   # shared bp1 == bp2
de1_dat <- .extract_lp(results_boot_val_bp1, "death")
de2_dat <- .extract_lp(results_boot_val_bp2, "death")

cidx_re  <- .run_cidx_boot(re_dat,  HZ_CIDX, B_CIDX, CIDX_SEED)
cidx_de1 <- .run_cidx_boot(de1_dat, HZ_CIDX, B_CIDX, CIDX_SEED)
cidx_de2 <- .run_cidx_boot(de2_dat, HZ_CIDX, B_CIDX, CIDX_SEED)

cat("== READMISSION (best_perf1 == best_perf2) | Uno's C, bootstrap B=", B_CIDX, " ==\n", sep="")
print(cidx_re,  row.names = FALSE)
cat("\n== DEATH best_perf1 (Full PH) | Uno's C ==\n");  print(cidx_de1, row.names = FALSE)
cat("\n== DEATH best_perf2 (SHAP)   | Uno's C ==\n");   print(cidx_de2, row.names = FALSE)
cat("\nUno's C with IPCW (timewt = n/G2); 95% CI = nonparametric bootstrap of the",
    "held-out test patients (model frozen, no refit), pooled over imputations.\n")

# ============================ IBS + combined C/IBS + null IBS =========================
# Per-window IBS with the SAME patient bootstrap and seed (CIDX_SEED) as Uno's C above, so
# C and IBS report PAIRED CIs. Model frozen, no refit; reuses raw_predictions. IBS = time-
# averaged IPCW Brier over [3, t]; IBS(3) is computed but undefined (single grid point) and
# excluded from the table. The null (intercept-only) IBS uses the same bootstrap and
# estimand as a skill reference (skill = 1 - IBS / IBS0).
source(file.path(if (exists("project_root")) project_root else getwd(),
                 "cons/_alt_scripts/ibs_window_bootstrap_holdout.R"))
IBS_READMIT_METHOD <- "ipcw"   # reproduces the reported readmission IBS (death as censoring)

ibs_bp1 <- ibs_window_bootstrap_core(results_boot_val_bp1, B = B_CIDX, seed = CIDX_SEED,
             eval_times = c(3, HZ_CIDX), readmit_method = IBS_READMIT_METHOD, verbose = FALSE)
ibs_bp2 <- ibs_window_bootstrap_core(results_boot_val_bp2, B = B_CIDX, seed = CIDX_SEED,
             eval_times = c(3, HZ_CIDX), readmit_method = IBS_READMIT_METHOD, verbose = FALSE)

.combine_c_ibs <- function(cidx_tab, ibs_df, which_risk) {
  ci  <- setNames(cidx_tab$Uno_C, cidx_tab$metric)
  d   <- ibs_df[ibs_df$risk == which_risk & is.finite(ibs_df$mean), ]
  nul <- attr(ibs_df, "null"); nul <- nul[nul$risk == which_risk & is.finite(nul$mean), ]
  fmt <- function(p, l, h) sprintf("%.4f (%.4f, %.4f)", p, l, h)
  ib  <- setNames(fmt(d$point, d$q025, d$q975),       as.character(d$horizon))
  i0  <- setNames(fmt(nul$point, nul$q025, nul$q975), as.character(nul$horizon))
  sk  <- setNames(sprintf("%.3f", 1 - d$point / nul$point), as.character(d$horizon))
  ib["Global"] <- ib["60"]; i0["Global"] <- i0["60"]; sk["Global"] <- sk["60"]
  ord <- c("6", "12", "36", "60", "Global")
  data.frame(Horizon = ord,
             `Uno's C (95% CI)`  = unname(ci[ord]),
             `IBS (95% CI)`      = unname(ib[ord]),
             `IBS null (95% CI)` = unname(i0[ord]),
             `IBS skill`         = unname(sk[ord]),
             check.names = FALSE, stringsAsFactors = FALSE)
}
tab_re  <- .combine_c_ibs(cidx_re,  ibs_bp1, "readmission")
tab_de1 <- .combine_c_ibs(cidx_de1, ibs_bp1, "death")
tab_de2 <- .combine_c_ibs(cidx_de2, ibs_bp2, "death")

cat("\n== Combined C + IBS | READMISSION (best_perf1 == best_perf2) ==\n"); print(tab_re[,  1:3], row.names = FALSE)
cat("\n== Combined C + IBS | DEATH best_perf1 (Full PH) ==\n");            print(tab_de1[, 1:3], row.names = FALSE)
cat("\n== Combined C + IBS | DEATH best_perf2 (SHAP) ==\n");               print(tab_de2[, 1:3], row.names = FALSE)

.null_msg <- function(tab, lbl)
  paste0(lbl, " | ", paste(sprintf("%s: IBS0 %s, skill %s", tab$Horizon,
         tab$`IBS null (95% CI)`, tab$`IBS skill`), collapse = " | "))
message("Null-model IBS (intercept-only; same bootstrap and estimand) and skill = 1 - IBS/IBS0:")
message("  ", .null_msg(tab_re,  "Readmission(shared)"))
message("  ", .null_msg(tab_de1, "Death best_perf1"))
message("  ", .null_msg(tab_de2, "Death best_perf2"))

# tidy IBS table for saving (one row per risk/model/horizon, point + 95% bootstrap CI)
ibs_window <- rbind(
  cbind(model = "shared",     ibs_bp1[ibs_bp1$risk == "readmission", ]),
  cbind(model = "best_perf1", ibs_bp1[ibs_bp1$risk == "death", ]),
  cbind(model = "best_perf2", ibs_bp2[ibs_bp2$risk == "death", ]))
rownames(ibs_window) <- NULL
if (exists("out_dir"))
  utils::write.csv(rbind(cbind(model = "readmit_shared",   tab_re),
                         cbind(model = "death_best_perf1", tab_de1),
                         cbind(model = "death_best_perf2", tab_de2)),
                   file.path(out_dir, "holdout_c_ibs_combined.csv"), row.names = FALSE)

In [ ]:
#| label: holdout-ici-bootstrap-run
#| message: false
library(future); library(future.apply)
if (!inherits(future::plan(), "multisession")) future::plan(future::multisession, workers = ms())#future::plan(future::multisession, workers = 20)
options(future.globals.maxSize = 10 * 1024^3)

ICI_B <- 1000L   # readmission AJ is the slow part; ~10-15 min parallel, ~45 min serial
#default is seed 2125L for reproducibility, but can be set to NULL for a fresh bootstrap sample each run
ici_boot <- ici_bootstrap_holdout(models, train_list, val_list,
                                   times = c(6, 12, 36, 60), B = ICI_B,
                                   seed = 2125L,
                                   parallel = TRUE, verbose = TRUE)
utils::write.csv(ici_boot, file.path(out_dir, "pred23_holdout_ici_bootstrap.csv"), row.names = FALSE)


~ 53 minutes

In [ ]:
#| label: holdout-ici-bootstrap-table
#| message: false
#| results: asis
library(htmltools)

ici_epi_table <- function(ici_boot, digits = 4) {
  d <- as.data.frame(ici_boot)
  f <- function(e, l, h) ifelse(is.na(e), "\u2014",
        sprintf("%.*f (%.*f, %.*f)", digits, e, digits, l, digits, h))
  out <- data.frame(
    Outcome = ifelse(d$risk == "readmission", "Readmission", "Mortality"),
    Model = dplyr::recode(d$model,
      "readmit::shared"   = "SHAP-informed (shared)",
      "death::best_perf1" = "Full PH (best_perf1)",
      "death::best_perf2" = "SHAP 13-var (best_perf2)"),
    Horizon = d$horizon,
    ICI = f(d$ici, d$ici_lo, d$ici_hi),
    ECE = f(d$ece, d$ece_lo, d$ece_hi),
    OE  = ifelse(is.na(d$eo), "\u2014", sprintf("%.2f (%.2f, %.2f)", d$eo, d$eo_lo, d$eo_hi)),
    stringsAsFactors = FALSE, check.names = FALSE)
  out[order(factor(out$Outcome, c("Readmission", "Mortality")), out$Model, out$Horizon), ]
}

# Generic grouped browsable table (reuse for ipeval / threshold tables too)
.browsable_grouped <- function(df, group_col, body_cols, header_labels, caption, footnote,
                               font = "'Times New Roman', serif") {
  th <- sprintf("position:sticky;top:0;background:#f3f3f3;border-bottom:2px solid #ccc;padding:6px 10px;text-align:center;white-space:nowrap;font-weight:bold;")
  td <- "border-bottom:1px solid #eee;padding:4px 10px;white-space:nowrap;"
  gh <- "background:#e8eef5;font-weight:bold;padding:5px 10px;border-bottom:1px solid #ccc;"
  header <- htmltools::tags$thead(htmltools::tags$tr(
    lapply(header_labels, function(nm) htmltools::tags$th(style = th, nm))))
  groups <- split(df, factor(df[[group_col]], levels = unique(df[[group_col]])))
  rows <- list()
  for (g in names(groups)) {
    rows[[length(rows) + 1L]] <- htmltools::tags$tr(
      htmltools::tags$td(colspan = length(body_cols), style = gh, g))
    gd <- groups[[g]]
    for (i in seq_len(nrow(gd)))
      rows[[length(rows) + 1L]] <- htmltools::tags$tr(lapply(seq_along(body_cols), function(j)
        htmltools::tags$td(style = paste0(td, "text-align:", if (j == 1) "left" else "center", ";"),
                           as.character(gd[i, body_cols[j]]))))
  }
  htmltools::browsable(htmltools::tags$div(
    htmltools::tags$div(style = paste0("font-weight:bold;margin-bottom:6px;font-family:", font, ";"), caption),
    htmltools::tags$div(style = paste0("max-height:550px;overflow:auto;border:1px solid #ddd;font-family:", font, ";font-size:13px;"),
      htmltools::tags$table(style = "border-collapse:collapse;width:max-content;min-width:100%;",
        header, htmltools::tags$tbody(rows))),
    htmltools::tags$div(style = paste0("font-size:11px;color:#555;margin-top:6px;max-width:780px;font-family:", font, ";"), footnote)))
}

if (exists("ici_boot")) {
  tab <- ici_epi_table(ici_boot)
  tab$Model[duplicated(paste(tab$Outcome, tab$Model))] <- ""
  .browsable_grouped(
    tab, group_col = "Outcome",
    body_cols     = c("Model", "Horizon", "ICI", "ECE", "OE"),
    header_labels = c("Model", "Horizon, mo", "ICI (95% CI)", "ECE (95% CI)", "E:O (95% CI)"),
    caption  = "Held-out (20%) calibration indices with bootstrap 95% CI.",
    footnote = paste0("ICI, integrated calibration index; ECE, estimated calibration error; ",
      "E:O, mean predicted / observed risk. Readmission observed risk via Aalen-Johansen ",
      "(death competing); mortality via Kaplan-Meier. Lower ICI/ECE = better; E:O 1.0 = perfect."))
}


~ 30 minutes

In [ ]:
#| label: holdout-ici-bootstrap-table2
ici_boot |>
  dplyr::transmute(model, horizon,
    ICI  = sprintf("%.4f (%.4f-%.4f)", ici, ici_lo, ici_hi),
    ECE  = sprintf("%.4f (%.4f-%.4f)", ece, ece_lo, ece_hi),
    `E:O`= sprintf("%.2f (%.2f-%.2f)",  eo,  eo_lo,  eo_hi)) |>
  knitr::kable("markdown",
    caption = "Held-out 20%: calibration indices with bootstrap 95% CI (predictions fixed, validation rows resampled)")


In [ ]:
#| label: holdout-ici-bootstrap-plot
#| fig-width: 8
#| fig-height: 9
library(ggplot2); library(patchwork)
tnr <- "Times New Roman"

plot_ici_boot_epi <- function(ici_boot, tnr = "Times New Roman",
                              metric_titles = c(ici = "Integrated Calibration Index (ICI)",
                                                ece = "Estimated Calibration Error (ECE)",
                                                eo  = "Expected:Observed ratio (E:O)")) {
  d <- as.data.frame(ici_boot)
  mk <- function(m) data.frame(risk = d$risk, model = d$model, horizon = d$horizon, metric = m,
    value = d[[m]], lower = d[[paste0(m, "_lo")]], upper = d[[paste0(m, "_hi")]])
  long <- do.call(rbind, lapply(names(metric_titles), mk))
  long$outcome <- factor(ifelse(long$risk == "readmission", "Readmission", "Mortality"),
                         levels = c("Readmission", "Mortality"))
  long$model_lab <- factor(dplyr::case_when(
    long$model == "readmit::shared"   ~ "SHAP-informed (readmission)",
    long$model == "death::best_perf1" ~ "Full PH \u2014 best_perf1 (mortality)",
    long$model == "death::best_perf2" ~ "SHAP 13-var \u2014 best_perf2 (mortality)",
    TRUE ~ long$model),
    levels = c("SHAP-informed (readmission)", "Full PH \u2014 best_perf1 (mortality)",
               "SHAP 13-var \u2014 best_perf2 (mortality)"))
  cols <- c("SHAP-informed (readmission)" = "#2166AC",
            "Full PH \u2014 best_perf1 (mortality)" = "#B2182B",
            "SHAP 13-var \u2014 best_perf2 (mortality)" = "#E08214")
  shp  <- c("SHAP-informed (readmission)" = 21,
            "Full PH \u2014 best_perf1 (mortality)" = 24,
            "SHAP 13-var \u2014 best_perf2 (mortality)" = 22)
  theme_epi <- theme_classic(base_size = 13, base_family = tnr) +
    theme(legend.position = "bottom", legend.title = element_blank(), legend.key.width = unit(1.2, "lines"),
          strip.background = element_blank(), strip.text = element_text(face = "bold", size = 12),
          axis.title = element_text(face = "bold"), axis.text = element_text(color = "black"),
          panel.grid.major.y = element_line(color = "grey92", linewidth = 0.3),
          panel.spacing = unit(0.8, "lines"), plot.tag = element_text(face = "bold", size = 16, family = tnr))
  mk_panel <- function(m, ytitle, ref = NA_real_, show_x = FALSE, show_strip = FALSE, expand0 = FALSE) {
    dd <- long[long$metric == m, ]
    dd <- dd[order(dd$model_lab, dd$horizon), ]
    pd <- position_dodge(width = 2.5)
    g <- ggplot(dd, aes(horizon, value, color = model_lab, shape = model_lab, group = model_lab))
    if (is.finite(ref)) g <- g + geom_hline(yintercept = ref, linetype = "22", color = "grey45", linewidth = 0.4)
    g <- g + geom_errorbar(aes(ymin = lower, ymax = upper), width = 1.2, linewidth = 0.5, alpha = 0.85, position = pd) +
      geom_line(linewidth = 0.7, alpha = 0.9, position = pd) + geom_point(size = 2.5, fill = "white", stroke = 0.9, position = pd) +
      facet_wrap(~ outcome, nrow = 1, scales = "free_y") +
      scale_color_manual(values = cols, drop = FALSE) + scale_shape_manual(values = shp, drop = FALSE) +
      scale_x_continuous(breaks = c(6, 12, 36, 60)) + labs(x = "Months since discharge", y = ytitle) + theme_epi
    if (expand0) g <- g + expand_limits(y = 0)
    if (!show_strip) g <- g + theme(strip.text = element_blank())
    if (!show_x) g <- g + theme(axis.title.x = element_blank(), axis.text.x = element_blank(), axis.ticks.x = element_blank())
    g
  }
  pA <- mk_panel("ici", metric_titles["ici"], show_x = FALSE, show_strip = TRUE,  expand0 = TRUE)
  pB <- mk_panel("ece", metric_titles["ece"], show_x = FALSE, show_strip = FALSE, expand0 = TRUE)
  pC <- mk_panel("eo",  metric_titles["eo"],  ref = 1, show_x = TRUE, show_strip = FALSE)
  (pA / pB / pC) + plot_layout(guides = "collect") + plot_annotation(tag_levels = "A") &
    theme(legend.position = "bottom")
}

if (exists("ici_boot")) {
  g_ici <- plot_ici_boot_epi(ici_boot, tnr); print(g_ici)
  ggplot2::ggsave(file.path(figs_out, "holdout_calibration_indices_boot2_panel.tiff"), g_ici,
                  width = 17.8, height = 20, units = "cm", dpi = 600, compression = "lzw")
  ggplot2::ggsave(file.path(figs_out, "holdout_calibration_indices_boot2_panel.png"), g_ici,
                  width = 17.8, height = 20, units = "cm", dpi = 600)
}


In [ ]:
#| label: holdout-ici-bootstrap-plot3
#| fig-width: 8
#| fig-height: 6.5

library(ggplot2); library(patchwork)
tnr <- "Times New Roman"

plot_ici_boot_epi <- function(ici_boot, tnr = "Times New Roman",
                              metric_titles = c(ici = "Integrated Calibration Index (ICI)",
                                                ece = "Estimated Calibration Error (ECE)"),
                              dodge = 3) {
  d <- as.data.frame(ici_boot)
  mk <- function(m) data.frame(risk = d$risk, model = d$model, horizon = d$horizon, metric = m,
    value = d[[m]], lower = d[[paste0(m, "_lo")]], upper = d[[paste0(m, "_hi")]])
  long <- do.call(rbind, lapply(names(metric_titles), mk))
  long$outcome <- factor(ifelse(long$risk == "readmission", "Readmission", "Mortality"),
                         levels = c("Readmission", "Mortality"))
  long$model_lab <- factor(dplyr::case_when(
    long$model == "readmit::shared"   ~ "SHAP-informed (readmission)",
    long$model == "death::best_perf1" ~ "Full PH \u2014 best_perf1 (mortality)",
    long$model == "death::best_perf2" ~ "SHAP 13-var \u2014 best_perf2 (mortality)",
    TRUE ~ long$model),
    levels = c("SHAP-informed (readmission)", "Full PH \u2014 best_perf1 (mortality)",
               "SHAP 13-var \u2014 best_perf2 (mortality)"))
  cols <- c("SHAP-informed (readmission)" = "#2166AC",
            "Full PH \u2014 best_perf1 (mortality)" = "#B2182B",
            "SHAP 13-var \u2014 best_perf2 (mortality)" = "#E08214")
  shp  <- c("SHAP-informed (readmission)" = 21,
            "Full PH \u2014 best_perf1 (mortality)" = 24,
            "SHAP 13-var \u2014 best_perf2 (mortality)" = 22)
  pd <- position_dodge(width = dodge)
  theme_epi <- theme_classic(base_size = 13, base_family = tnr) +
    theme(legend.position = "bottom", legend.title = element_blank(), legend.key.width = unit(1.2, "lines"),
          strip.background = element_blank(), strip.text = element_text(face = "bold", size = 12),
          axis.title = element_text(face = "bold"), axis.text = element_text(color = "black"),
          panel.grid.major.y = element_line(color = "grey92", linewidth = 0.3),
          panel.spacing = unit(0.8, "lines"), plot.tag = element_text(face = "bold", size = 16, family = tnr))
  mk_panel <- function(m, ytitle, show_x = FALSE, show_strip = FALSE) {
    dd <- long[long$metric == m, ]
    g <- ggplot(dd, aes(horizon, value, color = model_lab, shape = model_lab, group = model_lab)) +
      geom_errorbar(aes(ymin = lower, ymax = upper), width = 2, linewidth = 0.5, alpha = 0.85, position = pd) +
      geom_line(linewidth = 0.7, alpha = 0.9, position = pd) +
      geom_point(size = 2.5, fill = "white", stroke = 0.9, position = pd) +
      facet_wrap(~ outcome, nrow = 1, scales = "free_y") +
      scale_color_manual(values = cols, drop = FALSE) + scale_shape_manual(values = shp, drop = FALSE) +
      scale_x_continuous(breaks = c(6, 12, 36, 60)) + expand_limits(y = 0) +
      labs(x = "Months since discharge", y = ytitle) + theme_epi
    if (!show_strip) g <- g + theme(strip.text = element_blank())
    if (!show_x) g <- g + theme(axis.title.x = element_blank(), axis.text.x = element_blank(), axis.ticks.x = element_blank())
    g
  }
  ms <- names(metric_titles); nM <- length(ms)
  panels <- lapply(seq_along(ms), function(k)
    mk_panel(ms[k], metric_titles[ms[k]], show_x = (k == nM), show_strip = (k == 1)))
  patchwork::wrap_plots(panels, ncol = 1) + plot_layout(guides = "collect") +
    plot_annotation(tag_levels = "A") & theme(legend.position = "bottom")
}

if (exists("ici_boot")) {
  g_ici <- plot_ici_boot_epi(ici_boot, tnr); print(g_ici)
  ggplot2::ggsave(file.path(figs_out, "holdout_calibration_indices_boot3_panel.tiff"), g_ici,
                  width = 17.8, height = 14, units = "cm", dpi = 600, compression = "lzw")
  ggplot2::ggsave(file.path(figs_out, "holdout_calibration_indices_boot3_panel.png"), g_ici,
                  width = 17.8, height = 14, units = "cm", dpi = 600)
}


In [ ]:
#| label: holdout-ici-bootstrap-plot4-epi
#| fig-width: 8
#| fig-height: 6.5
source(file.path(project_root, "cons/_alt_scripts/plot_calibration_indices_epi.R"))
g_cal_idx <- plot_calibration_indices_epi(
  ici_boot,
  tnr          = "Times New Roman",
  panels       = c("ici", "eo"),
  figs_out     = figs_out,
  save         = TRUE,
  emit_caption = TRUE,
  eo_band      = NULL,
  eo_breaks    = c(0.8, 1, 1.25, 1.5),
  lang         = "en",
  model_labels = c(
    "readmit::shared"   = "Readmission",
    "death::best_perf1" = "Mortality: all\npredictors",
    "death::best_perf2" = "Mortality: SHAP-\ninformed"
  )
)
print(g_cal_idx)
# caption prints below the chunk as a message; or place it explicitly:
message(attr(g_cal_idx, "caption"))

## Threshold-based metrics

In [ ]:
#| label: holdout-threshold-bootstrap-pre
#| message: false

if (!exists("project_root")) project_root <- gsub("/cons$", "", here::here())
source(file.path(project_root, "cons/_alt_scripts/validate_holdout_metrics.R"))


In [ ]:
#| label: holdout-threshold-bootstrap-run
#| message: false
library(future); library(future.apply)
if (!inherits(future::plan(), "multisession")) future::plan(future::multisession, workers = 20)
options(future.globals.maxSize = 10 * 1024^3)

# Plug-in point estimate + percentile bootstrap CI with G FROZEN: the IPCW censoring
# distribution G is estimated ONCE on the full sample and only patients are resampled.
# This keeps the plug-in coherent with its own interval; re-estimating G per resample left
# the point marginally outside the CI for rare-event metrics (e.g. death sensitivity).
thr_boot <- threshold_bootstrap_holdout(
  list(best_perf1 = results_boot_val_bp1, best_perf2 = results_boot_val_bp2),
  horizons  = c(6, 12, 36, 60),
  central   = "plugin",          # full-sample point estimate (unchanged)
  ci_method = "percentile",
  freeze_g  = TRUE,              # G estimated once on the full sample; patients resampled
  # default thresholds: readmission c(.10,.15,.20,.25,.30,.40); death c(.01,.02,.03,.05,.075,.10)
  B = 1000L, seed = 2125L, parallel = TRUE, verbose = TRUE)
utils::write.csv(thr_boot, file.path(out_dir, "pred23_holdout_threshold_bootstrap.csv"), row.names = FALSE)

# coherence check: the plug-in estimate must fall within its CI
viol <- 0L
for (m in c("Sens", "Spec", "PPV", "NPV")) {
  pt <- thr_boot[[m]]; lo <- thr_boot[[paste0(m, "_lo")]]; hi <- thr_boot[[paste0(m, "_hi")]]
  viol <- viol + sum(is.finite(pt) & is.finite(lo) & is.finite(hi) & (pt < lo - 1e-9 | pt > hi + 1e-9))
}
cat(sprintf("\n=== Threshold metrics (plug-in + percentile, frozen G); cells outside the CI: %d ===\n", viol))

In [ ]:
#| label: holdout-threshold-bootstrap-table
threshold_browsable <- function(body, outcome, caption = NULL, note = NULL) {
  tags <- htmltools::tags
  HTML <- htmltools::HTML
  tagList <- htmltools::tagList
  browsable <- htmltools::browsable

  cls <- paste0("thr-table-", as.integer(runif(1, 1e6, 9e6)))
  ncols <- ncol(body)

  rows <- list()
  k <- 1L

  for (g in unique(outcome)) {
    idx <- which(outcome == g)

    rows[[k]] <- tags$tr(
      class = "group-row",
      tags$td(colspan = ncols, g)
    )
    k <- k + 1L

    for (i in idx) {
      rows[[k]] <- tags$tr(
        lapply(seq_len(ncols), function(j) {
          al <- if (j == 1) "left" else if (j %in% c(2, 3)) "right" else "center"
          tags$td(style = paste0("text-align:", al, ";"), body[i, j, drop = TRUE])
        })
      )
      k <- k + 1L
    }
  }

  browsable(tagList(
    tags$style(HTML(sprintf("
      .%s {
        border-collapse: collapse;
        margin: 0 auto;
        font-family: 'Times New Roman', serif;
        font-size: 12px;
        line-height: 1.25;
      }
      .%s caption {
        caption-side: top;
        font-weight: bold;
        margin-bottom: 8px;
      }
      .%s th, .%s td {
        border-bottom: 1px solid #dddddd;
        padding: 4px 8px;
        white-space: nowrap;
      }
      .%s th {
        border-bottom: 2px solid #777777;
        font-weight: bold;
      }
      .%s tbody tr:nth-child(even):not(.group-row) {
        background-color: #fafafa;
      }
      .%s tbody tr:not(.group-row):hover {
        background-color: #f3f6fb;
      }
      .%s .group-row td {
        background-color: #f3f3f3;
        font-weight: bold;
        text-align: left;
        border-top: 1px solid #bbbbbb;
      }
      .%s-note {
        max-width: 900px;
        margin: 8px auto 0 auto;
        font-family: 'Times New Roman', serif;
        font-size: 11px;
        line-height: 1.3;
      }
    ", cls, cls, cls, cls, cls, cls, cls, cls, cls))),

    tags$table(
      class = cls,
      if (!is.null(caption)) tags$caption(caption),
      tags$thead(tags$tr(lapply(names(body), tags$th))),
      tags$tbody(rows)
    ),

    if (!is.null(note)) tags$div(class = paste0(cls, "-note"), note)
  ))
}

threshold_epi_table <- function(thr, digits = 2, metrics = c("Sens","Spec","PPV","NPV")) {
  d <- as.data.frame(thr)
  d <- d[order(factor(ifelse(d$risk=="readmission","Readmission","Mortality"), c("Readmission","Mortality")),
               d$model, d$horizon, d$threshold), ]
  f <- function(m) ifelse(is.na(d[[m]]), "\u2014",
        sprintf("%.*f (%.*f, %.*f)", digits, unname(d[[m]]),
                digits, unname(d[[paste0(m,"_lo")]]), digits, unname(d[[paste0(m,"_hi")]])))
  nm <- c(Sens="Sensitivity", Spec="Specificity", PPV="PPV", NPV="NPV")
  out <- data.frame(
    Outcome = ifelse(d$risk=="readmission","Readmission","Mortality"),
    Model = dplyr::recode(d$model, "readmit::shared"="SHAP-informed (shared)",
      "death::best_perf1"="Full PH (best_perf1)", "death::best_perf2"="SHAP 13-var (best_perf2)"),
    Horizon = d$horizon, Threshold = sprintf("%g%%", 100*d$threshold),
    check.names = FALSE, stringsAsFactors = FALSE)
  for (m in metrics) out[[nm[m]]] <- f(m)
  list(tab = out, grp = table(factor(out$Outcome, levels = unique(out$Outcome))))
}

tt <- threshold_epi_table(thr_boot)
out <- tt$tab

km <- paste(out$Outcome, out$Model)
kh <- paste(km, out$Horizon)

out$Model   <- ifelse(duplicated(km), "", out$Model)
out$Horizon <- ifelse(duplicated(kh), "", as.character(out$Horizon))

body <- out[, -1]
names(body)[1:3] <- c("Model", "Horizon, mo", "Threshold")


In [ ]:
#| label: holdout-threshold-bootstrap-table2

threshold_epi_table <- function(thr, digits = 2, metrics = c("Sens","Spec","PPV","NPV")) {
  d <- as.data.frame(thr)
  d <- d[order(factor(ifelse(d$risk=="readmission","Readmission","Mortality"), c("Readmission","Mortality")),
               d$model, d$horizon, d$threshold), ]
  f <- function(m) ifelse(is.na(d[[m]]), "\u2014",
        sprintf("%.*f (%.*f, %.*f)", digits, unname(d[[m]]),
                digits, unname(d[[paste0(m,"_lo")]]), digits, unname(d[[paste0(m,"_hi")]])))
  nm <- c(Sens="Sensitivity", Spec="Specificity", PPV="PPV", NPV="NPV")
  out <- data.frame(
    Outcome = ifelse(d$risk=="readmission","Readmission","Mortality"),
    Model = dplyr::recode(d$model, "readmit::shared"="SHAP-informed (shared)",
      "death::best_perf1"="Full PH (best_perf1)", "death::best_perf2"="SHAP 13-var (best_perf2)"),
    Horizon = d$horizon, Threshold = sprintf("%g%%", 100*d$threshold),
    check.names = FALSE, stringsAsFactors = FALSE)
  for (m in metrics) out[[nm[m]]] <- f(m)
  list(tab = out, grp = table(factor(out$Outcome, levels = unique(out$Outcome))))
}

tt <- threshold_epi_table(thr_boot)
out <- tt$tab

km <- paste(out$Outcome, out$Model)
kh <- paste(km, out$Horizon)

out$Model   <- ifelse(duplicated(km), "", out$Model)
out$Horizon <- ifelse(duplicated(kh), "", as.character(out$Horizon))

body <- out[, -1]
names(body)[1:3] <- c("Model", "Horizon, mo", "Threshold")

threshold_browsable(
  body = body,
  outcome = out$Outcome,
  caption = "Held-out (20%) threshold-dependent metrics with bootstrap 95% CI.",
  note = "Predicted risk = 1-S(t) (cause-specific Cox). Readmission: competing-risk IPCW (Aalen-Johansen-aware; deaths before t weighted as controls); mortality: standard IPCW (Kaplan-Meier). 95% CI from bootstrap resampling of the held-out set. Readmission model shared by both."
)

## Save and session info

In [ ]:
#| label: session-info
#| echo: true
#| error: true
#| message: true
#| paged.print: true

message(paste0("R library: ", Sys.getenv("R_LIBS_USER")))
message(paste0("Date: ",withr::with_locale(new = c('LC_TIME' = 'C'), code =Sys.time())))
message(paste0("Editor context: ", getwd()))
cat("quarto version: "); quarto::quarto_version()
sesion_info <- devtools::session_info()

tabla_pkg <- dplyr::select(
  tibble::as_tibble(sesion_info$packages),
  package,
  loadedversion,
  source
)

tabla_pkg <- tibble::rowid_to_column(tabla_pkg, var = "row_number")

names(tabla_pkg) <- c("Row number", "Package", "Version", "Source")

htmltools::browsable(
  htmltools::tags$div(
    style = "
      max-height: 420px;
      overflow: auto;
      border: 1px solid #ddd;
      font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
      font-size: 70%;
      line-height: 0.75em;
      width: 100%;
    ",
    htmltools::tags$caption(
      style = "
        caption-side: top;
        text-align: left;
        display: block;
        padding: 6px 4px;
        font-size: 120%;
        line-height: 1.2em;
      ",
      htmltools::em("R packages")
    ),
    htmltools::tags$table(
      style = "
        border-collapse: collapse;
        width: max-content;
        min-width: 100%;
        white-space: nowrap;
      ",
      htmltools::tags$thead(
        htmltools::tags$tr(
          lapply(names(tabla_pkg), function(nm) {
            htmltools::tags$th(
              style = "
                position: sticky;
                top: 0;
                z-index: 2;
                background: #f3f3f3;
                border-bottom: 1px solid #ccc;
                padding: 4px 8px;
                text-align: left;
                white-space: nowrap;
              ",
              nm
            )
          })
        )
      ),
      htmltools::tags$tbody(
        lapply(seq_len(nrow(tabla_pkg)), function(i) {
          htmltools::tags$tr(
            lapply(tabla_pkg[i, ], function(x) {
              htmltools::tags$td(
                style = "
                  border-bottom: 1px solid #eee;
                  padding: 3px 8px;
                  white-space: nowrap;
                ",
                as.character(x)
              )
            })
          )
        })
      )
    )
  )
)

In [ ]:
#| label: holdout-save
holdout_validation <- list(
  created    = as.character(Sys.time()),
  seed       = 2125L,
  split_file = "cons/_out/comb_split_seed2125_test20_mar26.parquet"
)

# --- discrimination + prediction error (+ raw predictions = DCA/threshold source) ---
if (exists("results_boot_val_bp1") && exists("results_boot_val_bp2"))
  holdout_validation$results_boot_val <- list(best_perf1 = results_boot_val_bp1,
                                              best_perf2 = results_boot_val_bp2)
if (exists("cindex_ibs_global")) holdout_validation$cindex_ibs_global <- cindex_ibs_global
# --- calibration (curves + pooled ICI/ECE/E:O point estimates) ---
if (exists("cal_readmit"))   holdout_validation$cal_readmit <- cal_readmit
if (exists("cal_death_bp1") && exists("cal_death_bp2"))
  holdout_validation$cal_death <- list(best_perf1 = cal_death_bp1, best_perf2 = cal_death_bp2)
# --- DCA ---
if (exists("dca_models_full")) holdout_validation$dca_models_full <- dca_models_full
if (exists("dca_nb_bp1") && exists("dca_nb_bp2"))
  holdout_validation$dca_nb <- list(best_perf1 = dca_nb_bp1, best_perf2 = dca_nb_bp2)
# --- ipeval bootstrap (AUC / Brier / O:E + CIs) ---
if (exists("ipeval_res"))        holdout_validation$ipeval <- ipeval_res
# --- NRI / IDI reclassification ---
if (exists("nri_idi_holdout"))   holdout_validation$nri_idi <- nri_idi_holdout
if (exists("nri_idi_model_table")) holdout_validation$nri_idi_table <- nri_idi_model_table
# --- ICI / ECE bootstrap ---
if (exists("ici_boot"))          holdout_validation$ici_boot <- ici_boot
# --- threshold-dependent metrics bootstrap (Sens/Spec/PPV/NPV) ---
if (exists("thr_boot"))          holdout_validation$thr_boot <- thr_boot
# --- IBS per-window bootstrap (point + 95% CI, paired with Uno's C) ---
if (exists("ibs_window"))        holdout_validation$ibs_window <- ibs_window
# --- provenance ---
if (exists("models")) holdout_validation$models <-
  lapply(models, function(m) lapply(m, function(f) paste(deparse(f), collapse = " ")))
if (exists("hd")) holdout_validation$checks <- hd$checks

out_dir <- file.path("data", "20241015_out", "pred23")
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

saveRDS(holdout_validation,
        file.path(out_dir, paste0("pred23_holdout_validation_", format(Sys.Date(), "%Y_%m_%d"), ".rds")))

cat("Saved to", file.path(out_dir, paste0("pred23_holdout_validation_", format(Sys.Date(), "%Y_%m_%d"), ".rds")), "\n")
cat("Top-level objects saved:\n"); print(names(holdout_validation))


In [ ]:
#| label: save-individual-plot-rds

plot_rds_dir <- file.path("data", "20241015_out", "pred23")
dir.create(plot_rds_dir, recursive = TRUE, showWarnings = FALSE)

plots_to_save <- list()

.add_existing <- function(save_name, object_name) {
  if (exists(object_name, inherits = TRUE)) {
    plots_to_save[[save_name]] <<- get(object_name, inherits = TRUE)
  } else {
    message("Skipping ", save_name, ": object not found: ", object_name)
  }
}

.add_expr <- function(save_name, expr) {
  p <- tryCatch(
    eval.parent(substitute(expr)),
    error = function(e) {
      message("Skipping ", save_name, ": ", conditionMessage(e))
      NULL
    }
  )
  if (!is.null(p)) plots_to_save[[save_name]] <<- p
  invisible(p)
}

# Plots that were printed but not assigned in earlier cells
if (exists("plot_metrics") && exists("results_boot_val_bp1")) {
  .add_expr("holdout_cindex_ibs_best_perf1", plot_metrics(results_boot_val_bp1$summary, NULL))
}
if (exists("plot_metrics") && exists("results_boot_val_bp2")) {
  .add_expr("holdout_cindex_ibs_best_perf2", plot_metrics(results_boot_val_bp2$summary, NULL))
}

if (exists("cal_curve_plot") && exists("cal_readmit")) {
  .add_expr("holdout_calibration_readmit_basic", cal_curve_plot(cal_readmit, NULL, "#2166AC"))
}
if (exists("cal_curve_plot") && exists("cal_death_bp1")) {
  .add_expr("holdout_calibration_death_best_perf1_basic", cal_curve_plot(cal_death_bp1, NULL, "#B2182B"))
}
if (exists("cal_curve_plot") && exists("cal_death_bp2")) {
  .add_expr("holdout_calibration_death_best_perf2_basic", cal_curve_plot(cal_death_bp2, NULL, "#B2182B"))
}

if (exists(".make_cal_panel") && exists("cal_readmit")) {
  .add_expr(
    "holdout_calibration_readmit_epi_panel",
    .make_cal_panel(cal_readmit, color = "#2166AC", x_lim = c(0, 0.4), x_by = 0.1)
  )
}

# Plot objects already assigned by the notebook
.add_existing("holdout_calibration_readmit_indices", "fig_r")
.add_existing("holdout_calibration_death_best_perf1_indices", "fig_d1")
.add_existing("holdout_calibration_death_best_perf2_indices", "fig_d2")

if (all(vapply(c("fig_r", "fig_d1", "fig_d2"), exists, logical(1), inherits = TRUE))) {
  fig_cal_indices_all <- .add_expr(
    "holdout_calibration_indices_three_panel",
    patchwork::wrap_plots(fig_r, fig_d1, fig_d2, ncol = 1) +
      patchwork::plot_annotation(
        tag_levels = "A",
        theme = ggplot2::theme(
          plot.tag = ggplot2::element_text(
            face = "bold",
            family = if (exists("tnr", inherits = TRUE)) get("tnr", inherits = TRUE) else ""
          )
        )
      )
  )
}

# Raw DCA panels (printed but not assigned)
if (exists("make_dca_panel_figure") && exists("dca_models_full")) {
  .add_expr("holdout_dca_readmit_raw",
    make_dca_panel_figure(dca_models_full$best_perf1$summary,
                          dca_models_full$best_perf2$summary,
                          outcome = "readmission", horizons = c(12, 36, 60)))
  .add_expr("holdout_dca_death_raw",
    make_dca_panel_figure(dca_models_full$best_perf1$summary,
                          dca_models_full$best_perf2$summary,
                          outcome = "death", horizons = c(12, 36, 60)))
}

# Standardized DCA panels
.add_existing("holdout_calibration_curves_three_panel", "g_cal_curves")
.add_existing("holdout_dca_abc", "p_dca_abc")
.add_existing("holdout_dca_readmit_std", "p_readmit_panel_hold")
.add_existing("holdout_dca_death_std",   "p_death_panel_hold")

# Epi calibration indices panel
.add_existing("holdout_calibration_indices_epi", "g_cal_idx")

# inside cell 59, after print(g_ici)
saveRDS(g_ici, file.path(plot_rds_dir, "holdout_ici_bootstrap_boot2_panel.rds"))

.add_existing("holdout_mortality_calibration_best_perf1_epi", "death_A")
.add_existing("holdout_mortality_calibration_best_perf2_epi", "death_B")
.add_existing("holdout_mortality_calibration_AB", "final_death")
.add_existing("holdout_ipeval_bootstrap", "g_ip")
.add_existing("holdout_ici_bootstrap", "g_ici")

.safe_name <- function(x) gsub("[^A-Za-z0-9_.-]+", "_", x)

saved_plot_rds <- vapply(names(plots_to_save), function(nm) {
  f <- file.path(plot_rds_dir, paste0(.safe_name(nm), ".rds"))
  saveRDS(plots_to_save[[nm]], f)
  f
}, character(1))

cat("Saved individual plot RDS files to:\n", normalizePath(plot_rds_dir, winslash = "/"), "\n")
print(saved_plot_rds)